# 03. MLP 학습: 데이터, 모델, 손실, 역전파, 평가

2차원 점 `(x, y)`의 부호 조합을 기준으로 두 클래스를 만든다. 직선 하나로 구분하기 어려운 XOR 형태이므로 은닉층이 있는 작은 신경망을 학습한다. 데이터는 직접 생성하며 다운로드가 필요 없다.

In [ ]:
import torch
from torch import nn

assert torch.cuda.is_available(), "GPU가 활성화된 Jupyter 커널을 선택하세요."
device = torch.device("cuda")
torch.manual_seed(42)

def labels(points):
    return ((points[:, 0] > 0) ^ (points[:, 1] > 0)).long()

x_train = torch.randn(1024, 2, device=device)
y_train = labels(x_train)
x_test = torch.randn(512, 2, device=device)
y_test = labels(x_test)

print("train:", x_train.shape, y_train.shape)
print("sample:", x_train[:5], "labels:", y_train[:5])

## 모델과 손실 함수

입력 특성은 2개, 은닉층의 뉴런은 `hidden`개, 출력 클래스는 2개다. 마지막 층의 출력은 logit이며 `CrossEntropyLoss`에 그대로 전달한다.

In [ ]:
hidden = 32
lr = 0.02
steps = 301

model = nn.Sequential(
    nn.Linear(2, hidden),
    nn.ReLU(),
    nn.Linear(hidden, 2),
).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
print(model)
print("학습 파라미터 수:", sum(p.numel() for p in model.parameters()))

## 학습과 평가

매 반복에서 `zero_grad`, `forward`, `loss`, `backward`, `step`을 확인한다. 학습에 사용하지 않은 테스트 점들로 정확도를 측정한다.

In [ ]:
torch.cuda.reset_peak_memory_stats()
for step in range(steps):
    model.train()
    optimizer.zero_grad()
    logits = model(x_train)
    loss = loss_fn(logits, y_train)
    loss.backward()

    if step == 0:
        print("첫 층 가중치 gradient 크기:", model[0].weight.grad.norm().item())

    optimizer.step()
    if step % 50 == 0:
        model.eval()
        with torch.inference_mode():
            test_logits = model(x_test)
            accuracy = (test_logits.argmax(dim=1) == y_test).float().mean().item()
        print(f"step={step:3d} loss={loss.item():.4f} test_accuracy={accuracy:.3f}")

print("최대 PyTorch 텐서 메모리(MiB):", torch.cuda.max_memory_allocated() / 2**20)

## 새로운 점을 넣어 추론

네 점의 좌표와 예측된 클래스를 비교한다.

In [ ]:
points = torch.tensor([[1., 1.], [1., -1.], [-1., 1.], [-1., -1.]], device=device)
model.eval()
with torch.inference_mode():
    predicted = model(points).argmax(dim=1)
print("points:", points.tolist())
print("expected:", labels(points).tolist())
print("predicted:", predicted.tolist())

**직접 바꿔볼 것:** `hidden`을 2, 8, 64로 변경하고, `lr`을 0.002, 0.02, 0.2로 바꿔 정확도와 손실을 비교한다. 값을 바꿀 때는 모델 생성 셀부터 다시 실행해 새 모델을 학습한다.